# 05. Model Preparation

## 5.1 Проверка финальной таблицы

In [1]:
import pandas as pd
import numpy as np

In [2]:
orders_analytics = pd.read_csv("../data/processed/orders_analytics.csv", parse_dates=["order_purchase_timestamp",
                                                                     "order_approved_at",
                                                                     "order_delivered_carrier_date",
                                                                     "order_delivered_customer_date",
                                                                     "order_estimated_delivery_date",
                                                                     "review_creation_date", 
                                                                     "review_answer_timestamp"])

In [3]:
orders_analytics[["order_purchase_timestamp", 
                  "order_approved_at", 
                  "order_delivered_carrier_date", 
                  "order_delivered_customer_date",
                  "order_estimated_delivery_date",
                  "review_creation_date", 
                  "review_answer_timestamp"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_purchase_timestamp       99441 non-null  datetime64[us]
 1   order_approved_at              99281 non-null  datetime64[us]
 2   order_delivered_carrier_date   97658 non-null  datetime64[us]
 3   order_delivered_customer_date  96476 non-null  datetime64[us]
 4   order_estimated_delivery_date  99441 non-null  datetime64[us]
 5   review_creation_date           98673 non-null  datetime64[us]
 6   review_answer_timestamp        98673 non-null  datetime64[us]
dtypes: datetime64[us](7)
memory usage: 5.3 MB


In [4]:
orders_analytics["purchase_year_month"].info()

<class 'pandas.Series'>
RangeIndex: 99441 entries, 0 to 99440
Series name: purchase_year_month
Non-Null Count  Dtype
--------------  -----
99441 non-null  str  
dtypes: str(1)
memory usage: 777.0 KB


In [5]:
#восстановление типа для purchase_year_month
orders_analytics["purchase_year_month"] = (
    pd.to_datetime(orders_analytics["purchase_year_month"])
    .dt.to_period("M")
)

In [6]:
orders_analytics["purchase_year_month"].dtype

period[M]

In [7]:
orders_analytics.shape

(99441, 64)

In [8]:
orders_analytics["bad_review"].value_counts()

bad_review
0.0    84179
1.0    14494
Name: count, dtype: int64

In [9]:
orders_analytics.duplicated().sum()

np.int64(0)

## 5.2. Проверка пропусков

In [10]:
#таблица пропусков
missing_table = pd.DataFrame({
    "missing_count": orders_analytics.isna().sum(),
    "missing_percent": orders_analytics.isna().mean() * 100
})


missing_table = (
    missing_table
    .sort_values(
        by="missing_percent",
        ascending=False
    )
)


missing_table = missing_table[
    missing_table["missing_count"] > 0
]
missing_table

,missing_count,missing_percent
carrier_handoff_time_days,3156,3.173741
order_delivered_customer_date,2965,2.981668
is_late_delivery,2965,2.981668
delivery_time_days,2965,2.981668
delivery_delay_days,2965,2.981668
delay_days,2965,2.981668
early_delivery_days,2965,2.981668
order_delivered_carrier_date,1783,1.793023
total_weight_g,791,0.795447
total_volume_cm3,791,0.795447


### 5.2.1. Решения по пропускам

In [11]:
missing_table["action"] = ""

In [12]:
missing_table.loc["carrier_handoff_time_days", "action"] = "create flag has_carrier_handoff + fill -1"
missing_table.loc["order_delivered_customer_date", "action"] = "create flag is_delivered + drop"
missing_table.loc["is_late_delivery", "action"] = "create flag is_delivered + fill 0"
missing_table.loc["delivery_time_days", "action"] = "create flag is_delivered + fill -1"
missing_table.loc["delivery_delay_days", "action"] = "create flag is_delivered + fill -1"
missing_table.loc["delay_days", "action"] = "create flag is_delivered + fill -1"
missing_table.loc["early_delivery_days", "action"] = "create flag is_delivered + fill -1"
missing_table.loc["order_delivered_carrier_date", "action"] = "create flag has_carrier_delivery + drop"
missing_table.loc["total_weight_g", "action"] = "create flag has_product_dimensions + fill median"
missing_table.loc["total_volume_cm3", "action"] = "create flag has_product_dimensions + fill median"
missing_table.loc["payment_difference", "action"] = "create flag has_payment_difference + fill median"
missing_table.loc["abs_payment_difference", "action"] = "create flag has_payment_difference + fill median"
missing_table.loc["order_total", "action"] = "create flag has_order_financial_data, has_financial_missing + fill median"
missing_table.loc["total_price", "action"] = "create flag has_order_financial_data, has_financial_missing + fill median"
missing_table.loc["total_freight", "action"] = "create flag has_order_financial_data, has_financial_missing + fill median"
missing_table.loc["mean_freight", "action"] = "create flag has_order_financial_data, has_financial_missing + fill median"
missing_table.loc["mean_item_price", "action"] = "create flag has_order_financial_data, has_financial_missing + fill median"
missing_table.loc["freight_ratio", "action"] = "create flag has_order_financial_data, has_financial_missing + fill median"
missing_table.loc["max_item_price", "action"] = "create flag has_order_financial_data, has_financial_missing + fill median"
missing_table.loc["products_count", "action"] = "fill 0"
missing_table.loc["items_count", "action"] = "fill 0"
missing_table.loc["categories_count", "action"] = "fill 0"
missing_table.loc["sellers_count", "action"] = "fill 0"
missing_table.loc["has_multiple_sellers", "action"] = "fill 0"
missing_table.loc["has_multiple_products", "action"] = "fill 0"
missing_table.loc["has_unknown_category", "action"] = "fill 0"
missing_table.loc["unknown_category_items", "action"] = "fill 0"
missing_table.loc["review_answer_timestamp", "action"] = "exclude from ML (leakage)"
missing_table.loc["review_score", "action"] = "exclude from ML (leakage)"
missing_table.loc["review_id", "action"] = "exclude from ML (leakage)"
missing_table.loc["reviews_count", "action"] = "exclude from ML (leakage)"
missing_table.loc["had_multiple_reviews", "action"] = "exclude from ML (leakage)"
missing_table.loc["bad_review", "action"] = "drop"
missing_table.loc["review_creation_date", "action"] = "exclude from ML (leakage)"
missing_table.loc["has_review_comment", "action"] = "exclude from ML (leakage)"
missing_table.loc["has_review_title", "action"] = "exclude from ML (leakage)"
missing_table.loc["customer_lng", "action"] = "create flag has_customer_location + fill median"
missing_table.loc["customer_lat", "action"] = "create flag has_customer_location + fill median"
missing_table.loc["order_approved_at", "action"] = "create flag has_order_approved + drop from ML"
missing_table.loc["approval_time_days", "action"] = "fill -1"
missing_table.loc["used_installments", "action"] = "create flag has_payment_data + fill 0"
missing_table.loc["payment_total", "action"] = "create flag has_payment_data + fill 0"
missing_table.loc["main_payment_type", "action"] = "create flag has_payment_data + fill unknown"
missing_table.loc["max_installments", "action"] = "create flag has_payment_data + fill 0"
missing_table.loc["payments_count", "action"] = "create flag has_payment_data + fill 0"
missing_table.loc["has_multiple_payment_types", "action"] = "create flag has_payment_data + fill 0"
missing_table.loc["payment_types_count", "action"] = "create flag has_payment_data + fill 0"

In [13]:
missing_table

,missing_count,missing_percent,action
carrier_handoff_time_days,3156,3.173741,create flag has_carrier_handoff + fill -1
order_delivered_customer_date,2965,2.981668,create flag is_delivered + drop
is_late_delivery,2965,2.981668,create flag is_delivered + fill 0
delivery_time_days,2965,2.981668,create flag is_delivered + fill -1
delivery_delay_days,2965,2.981668,create flag is_delivered + fill -1
delay_days,2965,2.981668,create flag is_delivered + fill -1
early_delivery_days,2965,2.981668,create flag is_delivered + fill -1
order_delivered_carrier_date,1783,1.793023,create flag has_carrier_delivery + drop
total_weight_g,791,0.795447,create flag has_product_dimensions + fill median
total_volume_cm3,791,0.795447,create flag has_product_dimensions + fill median


### 5.2.2 Доставка

Причина пропусков:
Отсутствует дата передачи заказа перевозчику.
Пропуск означает отсутствие события, поэтому создаём бинарный признак
и заменяем значение на -1.

In [14]:
orders_analytics["has_carrier_handoff"] = (
    orders_analytics["carrier_handoff_time_days"]
    .notna()
    .astype(int)
)

In [15]:
orders_analytics["carrier_handoff_time_days"] = (
    orders_analytics["carrier_handoff_time_days"]
    .fillna(-1)
)

In [16]:
orders_analytics["carrier_handoff_time_days"].isna().sum()

np.int64(0)

In [17]:
orders_analytics["has_carrier_handoff"].value_counts()

has_carrier_handoff
1    96285
0     3156
Name: count, dtype: int64

Причина пропусков:
Отсутствует дата доставки заказа покупателю.

In [18]:
orders_analytics["is_delivered"] = (
    orders_analytics["order_delivered_customer_date"]
    .notna()
    .astype(int)
)

In [19]:
orders_analytics = orders_analytics.drop(
    columns="order_delivered_customer_date"
)

In [20]:
temporal_characteristics = ["delivery_time_days", "delivery_delay_days", "delay_days", "early_delivery_days"]

for characteristic in temporal_characteristics:
    orders_analytics[characteristic] = (
    orders_analytics[characteristic]
    .fillna(-1)
)

orders_analytics["is_late_delivery"] = orders_analytics["is_late_delivery"].fillna(0)

Причина пропусков в столбце `order_delivered_carrier_date` - отсутствует дата передачи заказа перевозчику.

In [21]:
orders_analytics["has_carrier_delivery"] = (
    orders_analytics["order_delivered_carrier_date"]
    .notna()
    .astype(int)
)

In [22]:
orders_analytics = orders_analytics.drop(
    columns="order_delivered_carrier_date"
)

### 5.2.3. Вес и объём

Причина пропусков `total_weight_g` и `totak_volume_cm3` - отсутствие данных по товару.

In [23]:
orders_analytics["has_product_dimensions"] = (
    orders_analytics[
        [
            "total_weight_g",
            "total_volume_cm3"
        ]
    ]
    .notna()
    .all(axis=1)
    .astype(int)
)

In [24]:
for col in [
    "total_weight_g",
    "total_volume_cm3"
]:
    orders_analytics[col] = (
        orders_analytics[col]
        .fillna(
            orders_analytics[col].median()
        )
    )

### 5.2.4. Финансовые признаки

#### Стоимость заказа

In [25]:
financial_features = [
    "order_total",
    "total_price",
    "total_freight",
    "mean_freight",
    "mean_item_price",
    "freight_ratio",
    "max_item_price"
]

orders_analytics["has_order_financial_data"] = (
    orders_analytics[financial_features]
    .notna()
    .any(axis=1)
    .astype(int)
)

orders_analytics["has_financial_missing"] = (
    orders_analytics[financial_features]
    .isna()
    .any(axis=1)
    .astype(int)
)

In [26]:
for col in financial_features:
    orders_analytics[col] = (
        orders_analytics[col]
        .fillna(
            orders_analytics[col].median()
        )
    )

#### Платёжные признаки

In [27]:
orders_analytics["has_payment_difference"] = orders_analytics["payment_difference"].notna().astype(int)

In [28]:
for col in [
    "payment_difference",
    "abs_payment_difference"
]:
    orders_analytics[col] = (
        orders_analytics[col]
        .fillna(
            orders_analytics[col].median()
        )
    )

### 5.2.5. Количество товаров

Причина пропусков `products_count`, `items_count`, `sellers_count`, `categories_count` - отсутствие информации о товарах.

In [29]:
cols = [
"products_count",
"items_count",
"categories_count",
"sellers_count"
]


orders_analytics[cols] = (
    orders_analytics[cols]
    .fillna(0)
)

### 5.2.6. Boolean признаки

In [30]:
cols = [
"has_multiple_sellers",
"has_multiple_products",
"has_unknown_category"
]


orders_analytics[cols] = (
    orders_analytics[cols]
    .fillna(0)
)

### 5.2.7. unknown_category_items

In [31]:
orders_analytics["unknown_category_items"] = (
    orders_analytics["unknown_category_items"]
    .fillna(0)
)

### 5.2.8. Отзывы

Причина пропусков: у заказа нет отзыва.

In [32]:
orders_analytics = (
    orders_analytics
    .dropna(subset=["bad_review"])
)

In [33]:
review_leakage_features = [
    "review_score",
    "review_id",
    "review_creation_date",
    "review_answer_timestamp",
    "has_review_comment",
    "has_review_title",
    "reviews_count",
    "had_multiple_reviews"
]

orders_analytics = orders_analytics.drop(
    columns=review_leakage_features
)

### 5.2.9. География

In [34]:
orders_analytics["has_customer_location"] = (
    orders_analytics[
        [
            "customer_lng",
            "customer_lat"
        ]
    ]
    .notna()
    .all(axis=1)
    .astype(int)
)

In [35]:
for col in [
    "customer_lng",
    "customer_lat"
]:
    orders_analytics[col] = (
        orders_analytics[col]
        .fillna(
            orders_analytics[col].median()
        )
    )

### 5.2.10. order_approved_at

In [36]:
orders_analytics["has_order_approved"] = orders_analytics["order_approved_at"].notna().astype(int)

In [37]:
orders_analytics["approval_time_days"] = (
    orders_analytics["approval_time_days"]
    .fillna(-1)
)

In [38]:
orders_analytics = orders_analytics.drop(columns="order_approved_at")

### 5.2.11. Платёжные признаки

In [39]:
orders_analytics["has_payment_data"] = (
    orders_analytics["payment_total"]
    .notna()
    .astype(int)
)

In [40]:
payment_features = [
    "payment_total",
    "max_installments",
    "payments_count",
    "payment_types_count",
    "used_installments",
    "has_multiple_payment_types"
]

orders_analytics[payment_features] = (
    orders_analytics[payment_features]
    .fillna(0)
)

In [41]:
orders_analytics["main_payment_type"] = (
    orders_analytics["main_payment_type"]
    .fillna("unknown")
)

### 5.2.12. Проверка NaN

In [42]:
orders_analytics.isna().sum().sum()

np.int64(0)

In [43]:
orders_analytics.duplicated().sum()

np.int64(0)

In [44]:
orders_analytics.info()

<class 'pandas.DataFrame'>
Index: 98673 entries, 0 to 99440
Data columns (total 63 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   order_id                        98673 non-null  str           
 1   customer_id                     98673 non-null  str           
 2   order_status                    98673 non-null  str           
 3   order_purchase_timestamp        98673 non-null  datetime64[us]
 4   order_estimated_delivery_date   98673 non-null  datetime64[us]
 5   customer_unique_id              98673 non-null  str           
 6   customer_zip_code_prefix        98673 non-null  int64         
 7   customer_city                   98673 non-null  str           
 8   customer_state                  98673 non-null  str           
 9   customer_lat                    98673 non-null  float64       
 10  customer_lng                    98673 non-null  float64       
 11  items_count       

## 5.3. Проверка категориальных признаков

In [45]:
categorical_features = (
    orders_analytics
    .select_dtypes(include="object")
    .columns
)

categorical_features

C:\Users\liza_\AppData\Local\Temp\ipykernel_17592\3969331866.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include="object")


Index(['order_id', 'customer_id', 'order_status', 'customer_unique_id',
       'customer_city', 'customer_state', 'main_payment_type',
       'is_late_delivery'],
      dtype='str')

In [46]:
orders_analytics["is_late_delivery"].value_counts()

is_late_delivery
False    91011
True      7662
Name: count, dtype: int64

### 5.3.1. Удаление ID

In [47]:
id_features = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

orders_analytics = orders_analytics.drop(
    columns=id_features
)

In [48]:
categorical_features = (
    orders_analytics
    .select_dtypes(include=["object", "string"])
    .columns
)

categorical_features

Index(['order_status', 'customer_city', 'customer_state', 'main_payment_type',
       'is_late_delivery'],
      dtype='str')

### Вывод

После проверки категориальных признаков были обнаружены следующие признаки типа object/string:

- order_id
- customer_id
- customer_unique_id
- order_status
- customer_city
- customer_state
- main_payment_type

Поля order_id, customer_id и customer_unique_id были удалены, так как являются идентификаторами и не несут полезной информации для модели. Их использование может привести к запоминанию конкретных объектов вместо выявления общих закономерностей.

После удаления идентификаторов были выделены следующие категориальные признаки:

- order_status
- customer_city
- customer_state
- main_payment_type

Дополнительно проверен признак is_late_delivery. Несмотря на то, что он попадает в выборку категориальных признаков, фактически он является бинарным числовым признаком (True/False), поэтому дополнительного кодирования не требует.

### 5.3.2. Приведение is_late_delivery к int

In [49]:
orders_analytics["is_late_delivery"] = (
    orders_analytics["is_late_delivery"]
    .astype(int)
)

In [50]:
categorical_features = (
    orders_analytics
    .select_dtypes(include=["object", "string"])
    .columns
)

categorical_features

Index(['order_status', 'customer_city', 'customer_state', 'main_payment_type'], dtype='str')

### 5.3.3 Удаление высококардинальных категориальных признаков

In [51]:
orders_analytics.select_dtypes(
    include=["object", "string"]
).nunique()

order_status            8
customer_city        4117
customer_state         27
main_payment_type       6
dtype: int64

In [52]:
orders_analytics = orders_analytics.drop(
    columns=["customer_city"]
)

Признак customer_city содержит 4117 уникальных значений.
Использование OneHotEncoding приведёт к значительному увеличению размерности данных и повышенному риску переобучения.

Так как признак customer_state содержит обобщённую географическую информацию, customer_city был удалён.

## 5.4 Проверка временных признаков и удаление сырых дат

In [53]:
orders_analytics.select_dtypes(
    include=["datetime"]
).columns

Index(['order_purchase_timestamp', 'order_estimated_delivery_date'], dtype='str')

In [54]:
orders_analytics = orders_analytics.drop(
    columns=["order_estimated_delivery_date", "order_purchase_timestamp"]
)

In [55]:
orders_analytics.select_dtypes(
    include=["datetime"]
).columns

Index([], dtype='str')

In [56]:
orders_analytics.select_dtypes(
    include=["period[M]"]
).columns

Index(['purchase_year_month'], dtype='str')

In [57]:
orders_analytics.drop(
    columns=["purchase_year_month"],
    inplace=True
)

In [58]:
orders_analytics.select_dtypes(
    include=["period[M]"]
).columns

Index([], dtype='str')

### Вывод

Проведена проверка временных признаков.

Сырые временные признаки не используются напрямую в модели, так как большинство алгоритмов машинного обучения не работают с типами datetime и period.

Перед обучением модели оставлены только производные признаки, полученные на этапе Feature Engineering:

- purchase_year
- purchase_month
- purchase_day_of_week
- purchase_hour
- purchase_is_weekend
- delivery_time_days
- delivery_delay_days

Сырые даты были удалены, так как их информация уже представлена в созданных числовых признаках.

## 5.5. Проверка константных признаков

In [63]:
constant_features = [
    col for col in orders_analytics.columns
    if orders_analytics[col].nunique() <= 1
]

constant_features

[]

## 5.6. Проверка типов

In [59]:
orders_analytics.dtypes.value_counts()

float64    35
int64      16
str         3
bool        2
Name: count, dtype: int64

### Вывод

После обработки данных проведена проверка типов признаков.

В итоговом датасете отсутствуют временные типы данных datetime, так как сырые даты были удалены после создания производных признаков.

Текущие типы признаков:

- float64 — 35 признаков;
- int64 — 16 признаков;
- str — 3 категориальных признака;
- bool — 2 бинарных признака.

Числовые признаки готовы к использованию в модели. 
Категориальные признаки будут закодированы на этапе подготовки обучающей выборки.

In [60]:
orders_analytics.isna().sum().sum()

np.int64(0)

## 5.7. Финальная проверка датасета

In [61]:
orders_analytics.columns

Index(['order_status', 'customer_zip_code_prefix', 'customer_state',
       'customer_lat', 'customer_lng', 'items_count', 'products_count',
       'sellers_count', 'categories_count', 'unknown_category_items',
       'total_price', 'mean_item_price', 'max_item_price', 'total_freight',
       'mean_freight', 'total_weight_g', 'total_volume_cm3',
       'has_unknown_category', 'has_multiple_sellers', 'has_multiple_products',
       'order_total', 'freight_ratio', 'payment_total', 'payments_count',
       'payment_types_count', 'max_installments', 'main_payment_type',
       'has_multiple_payment_types', 'used_installments', 'payment_difference',
       'abs_payment_difference', 'purchase_year', 'purchase_month',
       'purchase_day_of_week', 'purchase_hour', 'purchase_is_weekend',
       'approval_time_days', 'carrier_handoff_time_days',
       'invalid_approval_carrier_order', 'delivery_time_days',
       'estimated_delivery_time_days', 'delivery_delay_days',
       'is_late_delivery'

In [62]:
print("Размер:", orders_analytics.shape)

print(
    "Пропуски:",
    orders_analytics.isna().sum().sum()
)

print(
    "Дубликаты:",
    orders_analytics.duplicated().sum()
)

print(
    "Типы:",
    orders_analytics.dtypes.value_counts()
)

Размер: (98673, 56)
Пропуски: 0
Дубликаты: 0
Типы: float64    35
int64      16
str         3
bool        2
Name: count, dtype: int64


### Вывод

После обработки данных:

- пропуски отсутствуют;
- сырые даты удалены;
- идентификаторы удалены;
- категориальные признаки проверены;
- высококардинальный признак customer_city удалён;
- оставшиеся категориальные признаки подготовлены к дальнейшему кодированию;
- типы данных приведены к корректным значениям;
- подготовлен итоговый датасет для построения ML-моделей.

Размер итогового датасета:
98673 объектов и 55 признаков.

Следующим этапом является разделение данных на обучающую и тестовую выборки, кодирование категориальных признаков и построение базовых моделей.